In [25]:
import cv2
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from pathlib import Path

In [2]:
from utils.utils import *
from utils.config import Config

In [ ]:
cfg = Config(name='Dataset_Haze_Clutter', data='AMI-Echocardiogram')
print(cfg)

if cfg.data == 'EchoNet-Dynamic':
    ECHO_DIR    = Path(cfg.data_path / 'data_raw' / cfg.data / 'frames')
    LABEL_DIR   = None

elif cfg.data == 'AMI-Echocardiogram':
    ECHO_DIR    = Path('/nas2/yoonlea/GenAI_dataset/dataset')
    LABEL_DIR   = Path('/nas2/yoonlea/GenAI_dataset/Labeled_100')

SAVE_DIR = Path(cfg.data_path) / 'data_raw' / cfg.data

print(f'ECHO_DIR: {ECHO_DIR}')
print(f'LABEL_DIR: {LABEL_DIR}')
print(f'SAVE_DIR: {SAVE_DIR}')

----------------------------------------------------------------------
                        Config Details (Auto)                         
----------------------------------------------------------------------

[ Base Settings ]
  name                   : Dataset_Haze_Clutter
  data                   : AMI-Echocardiogram
  mkdir                  : False
  dtype                  : float64
  device                 : cpu
  multi_gpu              : False

[ Paths ]
  base_path              : /ds/mkseo/SM-Dehazing
  data_path              : /ds/mkseo/SM-Dehazing/data
  res_path               : /ds/mkseo/SM-Dehazing/res/Dataset_Haze_Clutter
  fig_path               : /ds/mkseo/SM-Dehazing/res/Dataset_Haze_Clutter/fig
  loss_path              : /ds/mkseo/SM-Dehazing/res/Dataset_Haze_Clutter/loss
  model_path             : /ds/mkseo/SM-Dehazing/res/Dataset_Haze_Clutter/model
----------------------------------------------------------------------
ECHO_DIR: /nas2/yoonlea/GenAI_dataset/dataset


In [ ]:
if cfg.data == 'AMI-Echocardiogram':
    echo_videos = [f for d_folder in ['data1', 'data2'] for f in globsort(ECHO_DIR / d_folder)]
    label_jsons = [f for l_folder in ['Results_Minseo', 'Results_Sohee'] for f in globsort(LABEL_DIR / l_folder)]
    # label_jsons = [p.with_name(f"{int(p.stem):09d}{p.suffix}") for p in label_jsons]

else: echo_videos = globsort(ECHO_DIR); label_jsons = None
print(f'Number of echo videos: {len(echo_videos)}')
print(*echo_videos[:5], sep='\n')
if label_jsons is not None: print(f'\nNumber of label jsons: {len(label_jsons)}'); print(*label_jsons[:5], sep='\n')

Number of echo videos: 1143
/nas2/yoonlea/GenAI_dataset/dataset/data1/000023935
/nas2/yoonlea/GenAI_dataset/dataset/data1/000036281
/nas2/yoonlea/GenAI_dataset/dataset/data1/000123482
/nas2/yoonlea/GenAI_dataset/dataset/data1/000202031
/nas2/yoonlea/GenAI_dataset/dataset/data1/000206189

Number of label jsons: 236
/nas2/yoonlea/GenAI_dataset/Labeled_100/Results_Minseo/126599.json
/nas2/yoonlea/GenAI_dataset/Labeled_100/Results_Minseo/182229.json
/nas2/yoonlea/GenAI_dataset/Labeled_100/Results_Minseo/198899.json
/nas2/yoonlea/GenAI_dataset/Labeled_100/Results_Minseo/279268.json
/nas2/yoonlea/GenAI_dataset/Labeled_100/Results_Minseo/469557.json


In [33]:
VIDEO_DIR   = SAVE_DIR / 'videos'
mk_dir([SAVE_DIR, VIDEO_DIR])

if cfg.data == 'AMI-Echocardiogram':
    filtered = []
    for label in tqdm(label_jsons, desc="Parsing JSON labels"):
        with open(label, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for ann in data['annotations']:
            if ann['modes'] == ['B'] and ann['notes'] == '':
                filtered = filtered + [ann]

    print(f'Number of filtered annotations: {len(filtered)}')

    pbar = tqdm(filtered, desc="Copying filtered videos")
    
    for ann in pbar:
        pid = f"{int(ann['id']):09d}"
        mk_dir(VIDEO_DIR / pid)
        
        video_dir = next((echo_dir for echo_dir in echo_videos if echo_dir.stem == pid), None)

        if video_dir is None:
            continue

        ori_video_path = video_dir / ann['file_name']
        save_video_path = VIDEO_DIR / pid / ann['file_name']
        
        pbar.set_postfix(file=ori_video_path.name)

        if ori_video_path.exists():
            # if save_video_path.exists():
            #     continue
                
            shutil.copy(ori_video_path, save_video_path)
        else:
            tqdm.write(f"⚠️ Original file not found: {ori_video_path}")

Parsing JSON labels: 100%|██████████| 236/236 [00:00<00:00, 1210.42it/s]


Number of filtered annotations: 2281


Copying filtered videos: 100%|██████████| 2281/2281 [00:55<00:00, 41.07it/s, file=100531518-67.mp4] 


In [ ]:
FRAME_DIR   = SAVE_DIR / 'frames'
mk_dir([SAVE_DIR, FRAME_DIR])

raw_videos = [raw_video for pid in listdir(VIDEO_DIR) for raw_video in globsort(VIDEO_DIR / pid)] if cfg.data == 'AMI-Echocardiogram' else echo_videos

used_frames, rows = [], []
pbar = tqdm(raw_videos, desc='Processing Echocardiograms')
for i, video in enumerate(pbar):
    frames = []
    cnt = 0

    cap = cv2.VideoCapture(video)
    cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frames.append(frame)
        cnt += 1

    cap.release()
    
    pbar.set_postfix(Video=video, Frames=cnt)

    idx = np.random.choice(cnt, 2, replace=False)
    
    for j, frame_idx in enumerate(idx):
        frame_path = FRAME_DIR / f'Patient_{(i+1):05d}_F{(j+1):03d}.png'
        f_img = cv2.cvtColor(frames[frame_idx], cv2.COLOR_BGR2RGB)
        f_img = Image.fromarray(f_img)

        f_img.save(str(frame_path), dpi=(500, 500))

        used_frames.append(f_img)

    rows.append({
        'Ori_ID': video.stem,
        'Study_ID': f'Patient_{(i+1):05d}',
        'Used_Frames': idx.tolist(),
    })

df = pd.DataFrame(rows)
df.to_csv(SAVE_DIR / 'metainfo.csv', index=False)
print(f'Saved frames: {len(used_frames)}')
df.head()

예상되는 넘파이 dtype: uint8
원본 Pixel Format: yuvj420p
채널당 Bit Depth: 8 bits
➡️ 이 파일은 평범한 8비트(uint8) 영상입니다.
